# SobaHealth Clinical Fine-Tune (Kaggle T4)

End-to-end notebook that fine-tunes `google/gemma-4-E2b-it` on **MedMCQA**, merges the LoRA adapter, converts to GGUF Q4_K_M, runs eval, and pushes the final GGUF to a private Hugging Face repo so your laptop can `wget` it.

> ## Heads up (May 2026): the GGUF cell currently produces a broken file for Gemma 4
>
> The training, eval, and adapter-upload cells all work. The **GGUF conversion**
> (`!bash training/convert_to_gguf.sh ...`) silently produces a header-less file
> because upstream `llama.cpp convert_hf_to_gguf.py` does not yet properly
> support Gemma 4's Matformer architecture (per-layer `key_length` and
> `feed_forward_length` get written as duplicate metadata keys and overwrite
> the header with zeros). See [`ggml-org/llama.cpp#22027`](https://github.com/ggml-org/llama.cpp/pull/22027)
> and [`#23047`](https://github.com/ggml-org/llama.cpp/issues/23047).
>
> **Until those land**, your run should still:
>
> 1. Train + eval (cells 1-10) - all good.
> 2. Push the **adapter** to HF (one of the upload cells). The adapter is the
>    valuable artefact and stays usable forever.
> 3. **Skip / expect failure** on the GGUF cell. The 880 MB file it uploads
>    cannot be loaded by Ollama (`Error: supplied file was not in GGUF format`).
>
> When upstream fixes either path (full GGUF or `convert_lora_to_gguf.py` for
> Gemma 4), nothing in this notebook needs to change - just re-run the GGUF
> cell, or use [`training/scripts/install_clinical_adapter.sh`](../scripts/install_clinical_adapter.sh)
> locally.

**Hardware**: Tesla T4 16 GB (free tier). Pin GPU = Yes, Internet = On.

**Before running**: in *Add-ons -> Secrets* add an `HF_TOKEN` secret with write access to your target repo.

## Two presets - pick one in cell 3

| `CONFIG_PATH` | Examples | Epochs | Total runtime | Expected MCQ gain |
|---|---|---|---|---|
| `training/configs/clinical_t4_fast.yaml` (default) | 1 500 | 1 | **~25 min** | +0.5 - 1.5 pp (smoke test) |
| `training/configs/clinical_t4.yaml` | 40 000 | 2 | ~7 hours | +5 - 10 pp (shipping) |

Start with the fast preset to verify the whole pipeline (train -> merge -> eval -> HF upload of adapter) works end-to-end. When you're happy, flip the toggle and run the full config before shipping.

## 1. Install dependencies

In [ ]:
!pip install -U -q unsloth
!pip install -q trl peft datasets accelerate bitsandbytes huggingface_hub sacrebleu pyyaml

## 2. Clone the SobaHealth repo for the training scripts

Skip if you uploaded the `training/` folder as a Kaggle dataset instead.

In [ ]:
import os
REPO_URL = "https://github.com/mrmayankmathur/SobaHealth.git"
REPO_DIR = "/kaggle/working/SobaHealth"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!ls training/

## 3. Load config + Hugging Face token

In [ ]:
import yaml, os
from kaggle_secrets import UserSecretsClient

# ============================================================================
# PICK YOUR PRESET (the only line you usually edit):
#
#   "training/configs/clinical_t4_fast.yaml"  ~25 min  - smoke / first run
#   "training/configs/clinical_t4.yaml"       ~7  h   - shipping
# ============================================================================
CONFIG_PATH = "training/configs/clinical_t4_fast.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# HF_REPO_ID comes from the config so the fast preset can't accidentally
# overwrite a shipping run. Override here if you need to point elsewhere.
HF_REPO_ID = cfg["huggingface"]["push_repo"]

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print("Config:    ", CONFIG_PATH)
print("Model:     ", cfg["model"]["hf_id"])
print("Train/Eval:", cfg["data"]["train_count"], "/", cfg["data"]["eval_count"])
print("HF target: ", HF_REPO_ID)

## 4. Prepare MedMCQA

In [ ]:
!python training/data/prepare_medmcqa.py --config {CONFIG_PATH}
!head -1 {cfg['data']['out_dir']}/train.jsonl

## 5. Load Gemma 4 E2B + attach LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = cfg["model"]["hf_id"],
    max_seq_length  = cfg["training"]["max_seq_length"],
    load_in_4bit    = cfg["model"]["load_in_4bit"],
    load_in_16bit   = cfg["model"]["load_in_16bit"],
    full_finetuning = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = cfg["lora"]["r"],
    target_modules = cfg["lora"]["target_modules"],
    lora_alpha     = cfg["lora"]["alpha"],
    lora_dropout   = cfg["lora"]["dropout"],
    bias           = "none",
    use_gradient_checkpointing = cfg["training"]["use_gradient_checkpointing"],
    random_state   = cfg["training"]["seed"],
    max_seq_length = cfg["training"]["max_seq_length"],
)
model.print_trainable_parameters()

## 6. Train

Wall-clock depends on the preset:
- **fast preset**: ~7-10 min on T4 (1 500 examples, 1 epoch, seq 512). Loss should drop from ~10 to ~1.5-2.5 by the end.
- **full preset**: ~3.5 hours / epoch on T4 (40 000 examples, 2 epochs, seq 1024). Loss should drop from ~10 to ~1.0-1.5 in epoch 1 and plateau around 0.8-1.2 in epoch 2.

If loss stalls above 3 something is off - check that `train.jsonl` is non-empty and the chat template is intact.

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

train_ds = load_dataset("json", data_files=f"{cfg['data']['out_dir']}/train.jsonl", split="train")
print("Train examples:", len(train_ds))
print("Sample:", train_ds[0]["text"][:300], "...")

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    args = SFTConfig(
        max_seq_length              = cfg["training"]["max_seq_length"],
        per_device_train_batch_size = cfg["training"]["per_device_train_batch_size"],
        gradient_accumulation_steps = cfg["training"]["gradient_accumulation_steps"],
        warmup_ratio                = cfg["training"]["warmup_ratio"],
        num_train_epochs            = cfg["training"]["num_train_epochs"],
        learning_rate               = cfg["training"]["learning_rate"],
        optim                       = cfg["training"]["optim"],
        logging_steps               = cfg["training"]["logging_steps"],
        save_strategy               = cfg["training"]["save_strategy"],
        seed                        = cfg["training"]["seed"],
        output_dir                  = cfg["training"]["output_dir"],
        dataset_text_field          = "text",
        dataset_num_proc            = 2,
    ),
)
trainer.train()

## 7. Save the LoRA adapter (for Option W later)

In [ ]:
model.save_pretrained(cfg["output"]["adapter_dir"])
tokenizer.save_pretrained(cfg["output"]["adapter_dir"])
!du -sh {cfg['output']['adapter_dir']}

## 8. Merge LoRA into base (for GGUF export)

In [ ]:
# Unsloth's helper does the merge + save in one call.
# `merged_16bit` keeps bfloat16 precision (smaller than fp32, identical quality).
model.save_pretrained_merged(
    cfg["output"]["merged_dir"],
    tokenizer,
    save_method="merged_16bit",
)
!du -sh {cfg['output']['merged_dir']}

## 9. Convert to GGUF Q4_K_M

Clones llama.cpp, runs `convert_hf_to_gguf.py`, then quantizes.

In [ ]:
!bash training/convert_to_gguf.sh {cfg['output']['merged_dir']} {cfg['output']['gguf_path']}
!ls -lh {cfg['output']['gguf_path']}

## 10. Quick eval (MCQ accuracy, base vs fine-tuned)

Runs held-out MedMCQA examples through both the base Gemma 4 E2B and the fine-tuned merged model and prints the accuracy delta. The merged model is reloaded from disk so we're testing the same artefact that will ship.

The fast preset caps this at 50 MCQ + 6 multilingual prompts to keep wall-clock under ~5 min; the full preset runs everything (~30 min).

In [ ]:
EVAL_REPORT = f"{cfg['training']['output_dir']}/eval_report.md"
MCQ_LIMIT  = cfg.get("eval", {}).get("mcq_limit", 0)
LANG_LIMIT = cfg.get("eval", {}).get("lang_limit", 0)

!python training/eval/run_all.py \
    --config {CONFIG_PATH} \
    --base-model {cfg['model']['hf_id']} \
    --tuned-model {cfg['output']['merged_dir']} \
    --eval-file {cfg['data']['out_dir']}/eval.jsonl \
    --multilingual-file training/eval/eval_set_indian_languages.jsonl \
    --mcq-limit {MCQ_LIMIT} \
    --lang-limit {LANG_LIMIT} \
    --out {EVAL_REPORT}
!cat {EVAL_REPORT}

## 11. Push GGUF + adapter + eval report to Hugging Face

In [ ]:
import os
from huggingface_hub import HfApi, create_repo

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO_ID, private=True, exist_ok=True, token=HF_TOKEN)

gguf_filename = os.path.basename(cfg["output"]["gguf_path"])

api.upload_file(
    path_or_fileobj = cfg["output"]["gguf_path"],
    path_in_repo    = gguf_filename,
    repo_id         = HF_REPO_ID,
)

api.upload_folder(
    folder_path  = cfg["output"]["adapter_dir"],
    path_in_repo = "adapter",
    repo_id      = HF_REPO_ID,
)

api.upload_file(
    path_or_fileobj = EVAL_REPORT,
    path_in_repo    = "eval_report.md",
    repo_id         = HF_REPO_ID,
)

print("Done. Pull on your laptop:")
print(f"  wget https://huggingface.co/{HF_REPO_ID}/resolve/main/{gguf_filename}")